# Lista 10 · Dataclasses

O equipamento deixa de ser um dicionário solto e passa a ser um **tipo**: uma
dataclass com campos conhecidos, que reclama na hora quando um nome é digitado
errado. A lista começa criando uma dataclass e um método, passa pelos padrões de
laço sobre uma lista de objetos e termina carregando um inventário em JSON.

A partir do exercício 05, as classes `Enlace` e `Equipamento` já vêm escritas no
esqueleto (marcadas com **JÁ ESCRITA**) — o trabalho é usá-las.

---

**Como usar este caderno:** cada exercício tem duas células. Na primeira,
escreva a sua solução no lugar do `# TODO`. A segunda tem os testes —
rode-a e ela diz se a sua função está correta. Não altere a célula de teste.

Se um teste falhar, o Python mostra um `AssertionError` apontando a linha:
é aquele caso específico que a sua função ainda não atende.

**Comece pela célula abaixo.** Ela cria os arquivos de exemplo que os
exercícios desta lista leem. Sem ela, os testes falham com
`FileNotFoundError`.

In [ ]:
"""Cria o inventário em JSON que os exercícios 09 e 10 leem.

Rode esta célula antes de tudo. Ela não baixa nada: escreve o arquivo direto na
pasta de trabalho da sessão.
"""

INVENTARIO = """{
  "coleta": "2026-03-02T23:59:00",
  "equipamentos": [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "ip": "10.0.1.10", "portas": "16", "em_servico": true},
    {"nome": "ONU-SUL-4512", "tipo": "ONU", "ip": "10.0.3.47", "portas": "1", "em_servico": true},
    {"nome": "OLT-NORTE-02", "tipo": "OLT", "ip": "10.0.2.10", "portas": "8", "em_servico": false},
    {"nome": "SWITCH-NORTE-02", "tipo": "SWITCH", "ip": "10.0.2.20", "portas": "48", "em_servico": true},
    {"nome": "OLT-SUL-03", "tipo": "OLT", "ip": "10.0.3.10", "portas": "24", "em_servico": true}
  ]
}
"""

with open("inventario.json", "w", encoding="utf-8") as arquivo:
    arquivo.write(INVENTARIO)

print("inventario.json criado")

### Exercício 01

Crie a dataclass `Enlace`, com os campos `nome` (texto), `potencia_rx` (número
real, em dBm) e `sensibilidade` (número real, com valor padrão `-27.0`).

```python
e = Enlace("ONU-SUL-4512", -21.4)
e.nome            # -> "ONU-SUL-4512"
e.sensibilidade   # -> -27.0
```

In [ ]:
from dataclasses import dataclass


# TODO: @dataclass e a classe Enlace com os três campos

In [ ]:
# Célula de teste — Exercício 01
e = Enlace("ONU-SUL-4512", -21.4)
assert e.nome == "ONU-SUL-4512"
assert e.potencia_rx == -21.4
assert e.sensibilidade == -27.0
outro = Enlace("ONU-LESTE-77", -24.0, sensibilidade=-28.0)
assert outro.sensibilidade == -28.0
assert Enlace("A", -20.0) == Enlace("A", -20.0)
print("Exercício 01: todos os testes passaram!")

### Exercício 02

Acrescente à `Enlace` o método `margem()`, que devolve a potência recebida menos a
sensibilidade, **arredondada com 2 casas**.

```python
Enlace("ONU-SUL-4512", -21.4).margem()   # -> 5.6
```

In [ ]:
from dataclasses import dataclass


@dataclass
class Enlace:
    nome: str
    potencia_rx: float
    sensibilidade: float = -27.0

    # TODO: def margem(self): ... (os campos se leem com self.potencia_rx)

In [ ]:
# Célula de teste — Exercício 02
assert Enlace("ONU-SUL-4512", -21.4).margem() == 5.6
assert Enlace("ONU-LESTE-77", -28.1).margem() == -1.1
assert Enlace("X", -25.0, sensibilidade=-25.0).margem() == 0.0
print("Exercício 02: todos os testes passaram!")

### Exercício 03

Acrescente à `Enlace` (que já tem `margem`) o método `situacao()`, que devolve:

- `"sem sinal"` se a margem for negativa;
- `"no limite"` se a margem for de 0 a 3 dB (inclusive);
- `"ok"` se for maior que 3 dB.

Use `self.margem()` dentro do método.

In [ ]:
from dataclasses import dataclass


@dataclass
class Enlace:
    nome: str
    potencia_rx: float
    sensibilidade: float = -27.0

    def margem(self):
        return round(self.potencia_rx - self.sensibilidade, 2)

    # TODO: def situacao(self): ...

In [ ]:
# Célula de teste — Exercício 03
assert Enlace("A", -21.4).situacao() == "ok"
assert Enlace("B", -28.1).situacao() == "sem sinal"
assert Enlace("C", -25.0).situacao() == "no limite"
assert Enlace("D", -27.0).situacao() == "no limite"
assert Enlace("E", -24.0).situacao() == "no limite"
print("Exercício 03: todos os testes passaram!")

### Exercício 04

Escreva `nao_fecham(enlaces)`, que recebe uma **lista de `Enlace`** e devolve a
lista dos **nomes** dos enlaces com margem negativa, na ordem da lista.

In [ ]:
from dataclasses import dataclass


@dataclass
class Enlace:
    """Um enlace óptico. JÁ ESCRITA."""
    nome: str
    potencia_rx: float
    sensibilidade: float = -27.0

    def margem(self):
        return round(self.potencia_rx - self.sensibilidade, 2)


def nao_fecham(enlaces):
    """Nomes dos enlaces com margem negativa."""
    # TODO: o padrão filtro, com e.margem() e e.nome
    pass

In [ ]:
# Célula de teste — Exercício 04
enlaces = [Enlace("A", -21.4), Enlace("B", -28.1), Enlace("C", -30.0), Enlace("D", -27.0)]
assert nao_fecham(enlaces) == ["B", "C"]
assert nao_fecham([]) == []
assert nao_fecham([Enlace("X", -29.0, sensibilidade=-30.0)]) == []
print("Exercício 04: todos os testes passaram!")

### Exercício 05

Com a dataclass `Equipamento` (já escrita), escreva `total_portas(inventario)`, que
devolve o total de portas dos equipamentos **em serviço**.

In [ ]:
from dataclasses import dataclass


@dataclass
class Equipamento:
    """Um equipamento do inventário. JÁ ESCRITA."""
    nome: str
    tipo: str
    ip: str
    portas: int
    em_servico: bool = True


def total_portas(inventario):
    """Total de portas dos equipamentos em serviço."""
    # TODO: acumulador, com e.em_servico e e.portas
    pass

In [ ]:
# Célula de teste — Exercício 05
inventario = [
    Equipamento("OLT-CENTRO-01", "OLT", "10.0.1.10", 16),
    Equipamento("ONU-SUL-4512", "ONU", "10.0.3.47", 1),
    Equipamento("OLT-NORTE-02", "OLT", "10.0.2.10", 8, em_servico=False),
    Equipamento("SWITCH-NORTE-02", "SWITCH", "10.0.2.20", 48),
    Equipamento("OLT-SUL-03", "OLT", "10.0.3.10", 24),
]
assert total_portas(inventario) == 89
assert total_portas([]) == 0
print("Exercício 05: todos os testes passaram!")

### Exercício 06

Escreva `de_dicionario(d)`, que recebe um dicionário como os do inventário em JSON
e devolve um `Equipamento`. Atenção: as portas chegam como **texto** e precisam
virar número; se a chave `em_servico` não existir, o equipamento está em serviço.

```python
de_dicionario({"nome": "OLT-CENTRO-01", "tipo": "OLT", "ip": "10.0.1.10", "portas": "16"})
# -> Equipamento(nome='OLT-CENTRO-01', tipo='OLT', ip='10.0.1.10', portas=16, em_servico=True)
```

In [ ]:
from dataclasses import dataclass


@dataclass
class Equipamento:
    """Um equipamento do inventário. JÁ ESCRITA."""
    nome: str
    tipo: str
    ip: str
    portas: int
    em_servico: bool = True


def de_dicionario(d):
    """Constrói um Equipamento a partir de um dicionário do JSON."""
    # TODO: Equipamento(nome=d["nome"], ..., portas=int(d["portas"]),
    #       em_servico=d.get("em_servico", True))
    pass

In [ ]:
# Célula de teste — Exercício 06
e = de_dicionario({"nome": "OLT-CENTRO-01", "tipo": "OLT", "ip": "10.0.1.10", "portas": "16"})
assert e == Equipamento("OLT-CENTRO-01", "OLT", "10.0.1.10", 16, True)
assert e.portas + 1 == 17
e = de_dicionario({"nome": "X", "tipo": "ONU", "ip": "1.1.1.1", "portas": "1", "em_servico": False})
assert e.em_servico is False
print("Exercício 06: todos os testes passaram!")

### Exercício 07

Escreva `maior_equipamento(inventario)`, que devolve o **objeto** `Equipamento` com
mais portas (a lista tem pelo menos um). Não use `max`: use o padrão extremo.

In [ ]:
from dataclasses import dataclass


@dataclass
class Equipamento:
    """Um equipamento do inventário. JÁ ESCRITA."""
    nome: str
    tipo: str
    ip: str
    portas: int
    em_servico: bool = True


def maior_equipamento(inventario):
    """O equipamento com mais portas."""
    # TODO: o candidato começa sendo inventario[0]; troque quando achar maior
    pass

In [ ]:
# Célula de teste — Exercício 07
inventario = [
    Equipamento("OLT-CENTRO-01", "OLT", "10.0.1.10", 16),
    Equipamento("ONU-SUL-4512", "ONU", "10.0.3.47", 1),
    Equipamento("OLT-NORTE-02", "OLT", "10.0.2.10", 8, em_servico=False),
    Equipamento("SWITCH-NORTE-02", "SWITCH", "10.0.2.20", 48),
    Equipamento("OLT-SUL-03", "OLT", "10.0.3.10", 24),
]
assert maior_equipamento(inventario).nome == "SWITCH-NORTE-02"
unico = Equipamento("X", "ONU", "1.1.1.1", 1)
assert maior_equipamento([unico]) is unico
print("Exercício 07: todos os testes passaram!")

### Exercício 08

Escreva `nomes_por_tipo(inventario)`, que devolve um dicionário **tipo → lista de
nomes**, na ordem do inventário (o agrupamento do capítulo 5).

In [ ]:
from dataclasses import dataclass


@dataclass
class Equipamento:
    """Um equipamento do inventário. JÁ ESCRITA."""
    nome: str
    tipo: str
    ip: str
    portas: int
    em_servico: bool = True


def nomes_por_tipo(inventario):
    """Dicionário tipo -> lista de nomes."""
    # TODO: se o tipo ainda não é chave, comece com lista vazia; depois append
    pass

In [ ]:
# Célula de teste — Exercício 08
inventario = [
    Equipamento("OLT-CENTRO-01", "OLT", "10.0.1.10", 16),
    Equipamento("ONU-SUL-4512", "ONU", "10.0.3.47", 1),
    Equipamento("OLT-NORTE-02", "OLT", "10.0.2.10", 8, em_servico=False),
    Equipamento("SWITCH-NORTE-02", "SWITCH", "10.0.2.20", 48),
    Equipamento("OLT-SUL-03", "OLT", "10.0.3.10", 24),
]
assert nomes_por_tipo(inventario) == {
    "OLT": ["OLT-CENTRO-01", "OLT-NORTE-02", "OLT-SUL-03"],
    "ONU": ["ONU-SUL-4512"],
    "SWITCH": ["SWITCH-NORTE-02"],
}
assert nomes_por_tipo([]) == {}
print("Exercício 08: todos os testes passaram!")

### Exercício 09

Escreva `carrega_inventario(caminho)`, que lê o arquivo JSON do inventário (com a
chave `"equipamentos"`) e devolve a **lista de `Equipamento`**. A função
`de_dicionario` já está escrita no esqueleto. A célula de preparo cria
`inventario.json`.

In [ ]:
import json
from dataclasses import dataclass


@dataclass
class Equipamento:
    """Um equipamento do inventário. JÁ ESCRITA."""
    nome: str
    tipo: str
    ip: str
    portas: int
    em_servico: bool = True


def de_dicionario(d):
    """Constrói um Equipamento a partir de um dicionário do JSON. JÁ ESCRITA."""
    return Equipamento(nome=d["nome"], tipo=d["tipo"], ip=d["ip"],
                       portas=int(d["portas"]), em_servico=d.get("em_servico", True))


def carrega_inventario(caminho):
    """Lista de Equipamento lida de um JSON de inventário."""
    # TODO: with open + json.load; para cada dicionário em dados["equipamentos"],
    #       append(de_dicionario(d))
    pass

In [ ]:
# Célula de teste — Exercício 09
inventario = carrega_inventario("inventario.json")
assert len(inventario) == 5
assert inventario[0] == Equipamento("OLT-CENTRO-01", "OLT", "10.0.1.10", 16, True)
assert inventario[2].em_servico is False
assert inventario[3].portas == 48
print("Exercício 09: todos os testes passaram!")

### Exercício 10

Escreva `relatorio(caminho)`, que carrega o inventário e devolve um texto com
**uma linha por equipamento em serviço** — nome em 18 colunas à esquerda, tipo em 8
colunas à esquerda, portas em 4 colunas à direita — e uma última linha com o total:

```
OLT-CENTRO-01     OLT       16
ONU-SUL-4512      ONU        1
SWITCH-NORTE-02   SWITCH    48
OLT-SUL-03        OLT       24
total: 89 portas em 4 equipamentos
```

`carrega_inventario` já está no esqueleto. Junte as linhas com `"\n".join(...)`.

In [ ]:
import json
from dataclasses import dataclass


@dataclass
class Equipamento:
    """Um equipamento do inventário. JÁ ESCRITA."""
    nome: str
    tipo: str
    ip: str
    portas: int
    em_servico: bool = True


def carrega_inventario(caminho):
    """Lista de Equipamento lida de um JSON de inventário. JÁ ESCRITA."""
    with open(caminho, encoding="utf-8") as arquivo:
        dados = json.load(arquivo)
    inventario = []
    for d in dados["equipamentos"]:
        inventario.append(Equipamento(nome=d["nome"], tipo=d["tipo"], ip=d["ip"],
                                      portas=int(d["portas"]),
                                      em_servico=d.get("em_servico", True)))
    return inventario


def relatorio(caminho):
    """Relatório dos equipamentos em serviço, com o total no fim."""
    # TODO: uma lista de linhas; para cada equipamento em serviço, uma f-string
    #       com {e.nome:<18}{e.tipo:<8}{e.portas:>4}; acumule portas e quantidade;
    #       no fim, a linha do total e "\n".join(linhas)
    pass

In [ ]:
# Célula de teste — Exercício 10
esperado = "\n".join([
    "OLT-CENTRO-01     OLT       16",
    "ONU-SUL-4512      ONU        1",
    "SWITCH-NORTE-02   SWITCH    48",
    "OLT-SUL-03        OLT       24",
    "total: 89 portas em 4 equipamentos",
])
assert relatorio("inventario.json") == esperado, relatorio("inventario.json")
print("Exercício 10: todos os testes passaram!")